# Upload the DATA209 videos to YouTube from Google Colab

**Before you start (one time, about 10 minutes)**
1. Upload the folder with the videos to **Google Drive**: unzip `DATA209_videos_part1/2/3` and put all `.mp4` files, the `.srt` files, the `thumbs` folder and `upload_manifest.json` in one Drive folder, for example `MyDrive/DATA209_upload`.
2. In [Google Cloud console](https://console.cloud.google.com) create a project, enable **YouTube Data API v3**, then open *APIs & Services → Credentials → Create credentials → OAuth client ID* and choose application type **TVs and Limited Input devices**. Copy the **Client ID** and **Client secret**.
3. Under *OAuth consent screen*, add your own Google account as a **test user**.

Then run the cells below from top to bottom. You sign in by typing a short code at google.com/device; nobody else sees your login.

Limits: a new project gets about 6 uploads a day, so run cell 5 once a day until all videos are done. It skips videos already uploaded.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
FOLDER = '/content/drive/MyDrive/DATA209_upload'   # change if you used another folder
import os; print(os.listdir(FOLDER)[:5])

In [ ]:
!pip -q install google-api-python-client google-auth requests

In [ ]:
import requests, time, getpass
CLIENT_ID = input('Client ID: ').strip()
CLIENT_SECRET = getpass.getpass('Client secret: ').strip()
SCOPE = 'https://www.googleapis.com/auth/youtube'   # needed for upload, captions, thumbnails, playlists
dev = requests.post('https://oauth2.googleapis.com/device/code', data={'client_id': CLIENT_ID, 'scope': SCOPE}).json()
print('1) Open', dev['verification_url'], '\n2) Enter the code:', dev['user_code'], '\n3) Sign in with your YouTube channel account and allow access.')

In [ ]:
# Run this after you finished the sign-in on google.com/device
while True:
    tok = requests.post('https://oauth2.googleapis.com/token', data={'client_id': CLIENT_ID, 'client_secret': CLIENT_SECRET,
        'device_code': dev['device_code'], 'grant_type': 'urn:ietf:params:oauth:grant-type:device_code'}).json()
    if 'access_token' in tok: break
    if tok.get('error') not in ('authorization_pending', 'slow_down'): raise SystemExit(tok)
    time.sleep(dev.get('interval', 5))
from google.oauth2.credentials import Credentials
from googleapiclient.discovery import build
creds = Credentials(tok['access_token'], refresh_token=tok.get('refresh_token'), token_uri='https://oauth2.googleapis.com/token',
                    client_id=CLIENT_ID, client_secret=CLIENT_SECRET, scopes=[SCOPE])
yt = build('youtube', 'v3', credentials=creds)
print('Signed in as channel:', yt.channels().list(part='snippet', mine=True).execute()['items'][0]['snippet']['title'])

In [ ]:
import json
from googleapiclient.http import MediaFileUpload
LIMIT = 6   # uploads per run (YouTube API daily quota for a new project)
done_path = os.path.join(FOLDER, 'uploaded.json')
done = json.load(open(done_path)) if os.path.exists(done_path) else {}
n = 0
for it in json.load(open(os.path.join(FOLDER, 'upload_manifest.json'))):
    if it['file'] in done or n >= LIMIT: continue
    status = {'privacyStatus': 'private', 'selfDeclaredMadeForKids': it['madeForKids'], 'containsSyntheticMedia': it['containsSyntheticMedia']}
    if it.get('publishAt'): status['publishAt'] = it['publishAt']   # goes public automatically at this time
    body = {'snippet': {'title': it['title'], 'description': it['description'], 'tags': it['tags'], 'categoryId': it['categoryId']}, 'status': status}
    req = yt.videos().insert(part='snippet,status', body=body, media_body=MediaFileUpload(os.path.join(FOLDER, it['file']), chunksize=-1, resumable=True))
    resp = None
    while resp is None: _, resp = req.next_chunk()
    vid = resp['id']
    yt.captions().insert(part='snippet', body={'snippet': {'videoId': vid, 'language': 'en', 'name': 'English'}},
                         media_body=MediaFileUpload(os.path.join(FOLDER, it['captions']))).execute()
    th = os.path.join(FOLDER, it.get('thumbnail', ''))
    if it.get('thumbnail') and os.path.exists(th):
        try: yt.thumbnails().set(videoId=vid, media_body=MediaFileUpload(th)).execute()
        except Exception as e: print('thumbnail not set (verify your channel at youtube.com/verify):', e)
    done[it['file']] = vid; json.dump(done, open(done_path, 'w'), indent=1)
    print('uploaded', it['title'], '->', 'https://youtu.be/' + vid); n += 1
print(n, 'uploaded this run;', len(done), 'in total')